# Warp on the NPE emulator — validated on the DB examples

HW-faithful warp emulation following `warp_m/HW_WARP.md` and the final format
decisions — all operands are **plain INTs on an MX container with a zero
exponent field** ("NOT MX INT BUT INT"):

| tensor | format | content |
|---|---|---|
| reference | **uint8** (round+clamp, NOT quantized to MX) | values 0..255, `mx-e4bm8`, exp = 0 |
| grid | **int16** `q = round(coord·2^fr)` per axis | **absolute** sample coordinate in pixels, `mx-e4bm16`, exp = 0 |
| TRC output | int in `[0, 2^(frx+fry)]` | corner indices + `w00..w11` (Σ = `2^(frx+fry)`), `mx-e4bm16`, exp = 0 |
| **output** | **int24 (relux)** | PCU: `acc >> (frx+fry)` round-half-up, 24-bit relux (int8 clamp only if you ask for it) |

Pipe: **TRC** (`trc_warp`: msb/lsb split `idx = q>>fr`, `frac = q & (2^fr−1)`;
corner addresses clipped to `[0, dim−2]` + the 4 weights) → **TRU-A**
(`tru_a_warp`: 2-line corner fetch) → **BCU** (4-term MAC, exponents all zero)
→ **PCU** (`build_warp_c_struct_arr`).

Why `mx-e4bm16` for the grid/weights: the TRC weight is an exact integer over
`2^(frx+fry)` (up to `2^14`), which does not fit the 8-bit mantissa lane.

Fraction bits follow the reference notebook (`EL_nb/new_warp_torch.ipynb`,
`grid_quantization`), per axis:

```
fr = grid_bit_width − floor(log2(ceil(max|coord_axis|))) − 2      (grid_bit_width = 16)
```

capped so `frx + fry ≤ 14` (weight lane), reducing both axes evenly on small
frames. No `auto_warp_fl` anymore — the packer derives it from the data.

Validated against `/home/eliyahul/ai_ws/MX_FORMAT/nndof_thetis_MX/warp_args`
(14 examples). Their exact float conventions were recovered first:

- 13 examples: **normalized-acT deltas, align_corners=True** ->
  `delta_px = flow · [(W−1)/2, (H−1)/2]`
- example 2: **pixel deltas** directly.
- channel 0 = x, channel 1 = y; same-size warp (out shape == in shape).

Assertions per example: (a) **exact integer equality** vs an integer golden
mirroring TRC → BCU → PCU on the same packed operands; (b) the packer follows
the notebook's `grid_quantization` bit-for-bit; (c) mse vs the stored float
output is reported next to its reference-rounding floor.

In [ ]:
import os
import sys
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import numpy as np
import torch

from npe_mx_emu import (
    CalibratedMXWarp, LayerMXQParams,
    build_warp_c_struct_arr, pack_warp_grid, pack_warp_reference,
)

DB = "/home/eliyahul/ai_ws/MX_FORMAT/nndof_thetis_MX/warp_args"
N_EX = 1
GRID_BITS = 16

# DB conventions (recovered): normalized-acT deltas for all examples except ex2
AC_T = [i for i in range(N_EX) if i != 2]


def load_example(i):
    """(ref, flow, out_db) float64 numpy from the stored DB."""
    ref = torch.load(f"{DB}/tenInputs/tenInput{i}.pt", map_location=torch.device('cpu')).double().numpy()   # (B,C,H,W)
    flow = torch.load(f"{DB}/tenFlows/tenFlow{i}.pt", map_location=torch.device('cpu')).double().numpy()    # (B,2,H,W)
    out_db = torch.load(f"{DB}/outputs/output{i}.pt", map_location=torch.device('cpu')).double().numpy()    # (B,C,H,W)
    return ref, flow, out_db


def acT_deltas(flow, W, H):
    scT = np.array([[(W - 1) / 2, (H - 1) / 2]]).reshape(1, 2, 1, 1)
    return flow * scT


def absolute_grid(flow, i, W, H):
    """Absolute sample coordinates [B, 2, H, W] in pixels (channel 0 = x, 1 = y)."""
    deltas = acT_deltas(flow, W, H) if i in AC_T else flow
    loc = np.dstack(np.meshgrid(np.arange(W), np.arange(H)))[None].transpose(0, 3, 1, 2)
    return loc + deltas

In [ ]:
# Run the existing DB example through BCU -> PCU (no golden verification).
ref, flow, out_db = load_example(0)
B, C, H, W = ref.shape
motion = acT_deltas(flow, W, H) if 0 in AC_T else flow
# Example producer conversion to signed Q8.8; warp consumes integer words.
grid = torch.tensor(np.floor(motion * 256 + 0.5).clip(-32768, 32767), dtype=torch.int16)
reference = torch.tensor(np.floor(ref + 0.5).clip(-128, 127), dtype=torch.int8)
qp = LayerMXQParams(x_cal=0, w_cal=0, out_cal=0)
warp = CalibratedMXWarp(channels=C, qparams=qp)
out = warp(reference, grid, frac_bits=(8, 8))
print("Output:", tuple(out.shape), out.dtype)
print(out)
